# Unitree D1 — Complete Guide

End-to-end reference: hardware, network, SDK, operation, troubleshooting, and research phases.

**Official docs:** https://support.unitree.com/home/en/developer/D1Arm_services

## 1. Hardware Reference

| Parameter | Value |
|-----------|-------|
| Model | D1-550 |
| DOF | 6 + 1 gripper |
| Arm length | 550 mm (670 mm with jaws) |
| Rated load | 500 g |
| Working radius | 550 mm |
| Motor type | Bus servo |
| Power | 24V 10A (15–48V), 240W |
| Communication | RJ45 Ethernet 100 Mbps |
| Control method | DDS subscription |
| Control cycle | 10 Hz |

### Joint Torques and Range of Motion

| Joint | Torque | Range |
|-------|--------|-------|
| J0 | 3.3 Nm | ±135° |
| J1 | 3.3 Nm | ±90° |
| J2 | 1.7 Nm | ±90° |
| J3 | 1.7 Nm | ±135° |
| J4 | 1.7 Nm | ±90° |
| J5 | 1.7 Nm | ±135° |
| J6 (gripper) | — | 0–65 mm stroke |

### Power Model

The D1 has **no power button**. Power ON = apply DC. Power OFF = remove DC.  
Motors disengage on power loss — arm will fall. This is expected industrial behavior.

## 2. Network Setup (One-Time)

The robot has a static IP and no DHCP. The host must be configured explicitly.

| Device | IP | Subnet |
|--------|----|--------|
| Robot | `192.168.123.100` | `/24` |
| Host | `192.168.123.10` | `/24` |
| NIC | `enx4cea4168e514` | — |

### Why This Matters

- If the host IP matches the robot IP, Linux routes traffic to loopback — `ping` succeeds but you're pinging yourself.
- NetworkManager can silently restore a cached profile and re-create the collision.
- DDS will silently select the wrong NIC (Wi-Fi or loopback) if not explicitly bound.

### Setup Commands

Remove any cached Ethernet profile:
```bash
nmcli connection delete "Wired connection 1"
```

Create a persistent profile for the robot NIC:
```bash
nmcli connection add type ethernet \
  ifname enx4cea4168e514 \
  con-name d1-robot \
  ipv4.method manual \
  ipv4.addresses 192.168.123.10/24 \
  ipv6.method ignore
```

Activate it:
```bash
nmcli connection up d1-robot
```

### Verify

```bash
ip addr show enx4cea4168e514     # must show inet 192.168.123.10/24
ip route get 192.168.123.100    # must show dev enx4cea4168e514 src 192.168.123.10
ip neigh show dev enx4cea4168e514  # must show 192.168.123.100 ... REACHABLE
```

If `ip route get` shows `dev lo` or a Wi-Fi interface, routing is wrong — re-check the profile.

## 3. SDK Build

`d1_sdk/src/` is already in this repo. The only external dependency is `unitree_sdk2` (includes CycloneDDS). On a new machine, install it once:

```bash
git clone https://github.com/unitreerobotics/unitree_sdk2
cd unitree_sdk2
mkdir build && cd build
cmake ..
sudo make install
```

Then build the examples:

```bash
cd d1_sdk
mkdir -p build && cd build
cmake ..
make -j$(nproc)
```

Executables produced in `d1_sdk/build/`:
- `get_arm_joint_angle` — telemetry stream, 10 Hz
- `joint_enable_control` — lock all joints
- `arm_zero_control` — move to mechanical zero
- `joint_angle_control` — move joint 5 to 60°
- `multiple_joint_angle_control` — move all joints to a preset pose

> On a new machine, run `./setup.sh` from the repo root — it handles network config, patches the NIC name into source, and builds automatically.


## 4. DDS Interface Binding

USB-to-Ethernet adapters get a non-standard name (`enx<MAC>`) rather than `eth0`. The SDK's `Init(0)` with no argument cannot find this name and **silently drops all commands** — no error, no output.

**Every program must pass the interface name explicitly:**
```cpp
ChannelFactory::Instance()->Init(0, "enx4cea4168e514");
```

All binaries in `d1_sdk/src/` already have this. If you write a new program, include it on the first line of `main()`.

If you swap the USB adapter or move to a different machine, find the new name with:
```bash
ip link show
```
and update the string in every source file.


## 5. Session Startup (Every Session)

1. Apply DC power to the robot.
2. Wait **60–90 seconds** for the robot to boot and DDS to become ready.
3. Verify network:
   ```bash
   ping 192.168.123.100
   ssh ubuntu@192.168.123.100   # password: 123
   ```
4. Verify robot services (optional):
   ```bash
   systemctl status marm_communication.service
   systemctl status marm_control.service
   systemctl status marm_controller.service
   systemctl status marm_subscripber.service
   ```
5. Run SDK executables from `d1_sdk/build/`.

> **Silence ≠ failure.** DDS discovery takes 30–40 s after the network is up. Wait before assuming something is wrong.


## 6. Command Reference

Run all executables from `d1_sdk/build/`:

```bash
cd d1_sdk/build
```

### SDK Executables

| Command | Effect |
|---------|--------|
| `./get_arm_joint_angle` | Stream joint angles at 10 Hz |
| `./joint_enable_control` | Enable (lock) all joints — no output, arm stiffens |
| `./arm_zero_control` | Move to mechanical zero |
| `./joint_angle_control` | Move joint 5 to 60° |
| `./multiple_joint_angle_control` | Move all joints to a preset pose |

### DDS Protocol (for custom programs)

Topics: `rt/arm_Command` (send) · `rt/arm_Feedback` (receive)

| funcode | Function | Example payload |
|---------|----------|-----------------|
| 1 | Single joint angle | `{"seq":4,"address":1,"funcode":1,"data":{"id":5,"angle":60,"delay_ms":0}}` |
| 2 | All joint angles | `{"seq":4,"address":1,"funcode":2,"data":{"mode":1,"angle0":0,"angle1":-60,"angle2":60,"angle3":0,"angle4":30,"angle5":0,"angle6":0}}` |
| 4 | Single joint enable/disable | `{"seq":4,"address":1,"funcode":4,"data":{"id":5,"mode":1}}` |
| 5 | All joints enable/disable | `{"seq":4,"address":1,"funcode":5,"data":{"mode":0}}` |
| 6 | Motor power switch | `{"seq":4,"address":1,"funcode":6,"data":{"power":1}}` |
| 7 | Return to zero | `{"seq":4,"address":1,"funcode":7}` |

`mode` for funcode 2: `0` = small smoothing (10 Hz), `1` = large smoothing (trajectory).


## 7. Troubleshooting

### DDS exception: `does not match an available interface` / `Failed to create domain explicitly`

**First check: is the robot powered?** This error does not say "no power" — it says the NIC is absent, which happens when the robot is off.

Resolution:
1. Check power cable.
2. Power on.
3. Wait 60–90 s.
4. Retry.

### Ping works but control does nothing

Check routing:
```bash
ip route get 192.168.123.100
```
If the output shows `dev lo` → host IP collides with robot IP. Redo network setup (Section 2).  
If it shows a Wi-Fi interface → no route for `192.168.123.0/24` via the robot NIC. Check the NIC profile.

### No telemetry output for 30–40 s

Normal. Wait for DDS discovery to stabilize. Do not restart.

### ARP verification (ground truth)

```bash
ip neigh show dev enx4cea4168e514
# must show: 192.168.123.100 lladdr <mac> REACHABLE
```
If the robot's MAC appears here, you are physically connected to a distinct device.

## 8. Phase 1 — Zero Reference Validation (CLOSED Dec 25)

### Findings

- Joint encoder feedback is stable within each zero run (≤ 0.1° jitter).
- `arm_zero_control` converges to a deterministic internal reference.
- Zero is **multi-valued across boots** for some joints (notably J4: ~0.7° vs ~72°).
- These are discrete branches — not continuous drift.
- No mechanical, electrical, or control instability detected.

### What Zero Means

Zero is defined by encoder index + hard-stop geometry + internal calibration constants.  
Mechanical zero ≠ all joint angles = 0. The wrist offset (~80°) is intentional.

### Constraints (Do Not Violate)

- Do not redefine or normalize zero.
- Do not subtract offsets in code.
- Do not average across zero branches.
- Zero is a reference only — define home pose separately.

Raw data: `logs/Phase-zero-cycle/` · Stats: `logs/Phase-zero-cycle/PHASE1_ZERO_STATS.md`

## 9. Phase 2 — Single-Joint Characterization (Next)

For joints 0→6, one at a time:
1. Start from zero.
2. Command +10°, wait 2–3 s.
3. Command back to zero.
4. Log: `./get_arm_joint_angle > ../../logs/phase2/joint_X_step_YY.txt`

**Observe:** correct joint moves · direction consistent · returns near reference · other joints still.

**Do NOT:** move multiple joints · use IK · adjust offsets · optimize.

**Close:** summarize per joint (moves? reversible? anomalies?), tag `phase2-joints-characterized`.


## 10. Roadmap

One phase active at a time. Full details and career rationale in `yearplan.md`.

| Phase | Description | Target | Status |
|-------|-------------|--------|--------|
| 0 | OS setup, bring-up, logging | — | ✅ Done |
| 1 | Zero & reference integrity | Dec 2025 | ✅ Done |
| 2 | Single-joint characterization | Jul 2026 | 🔜 Next |
| 3 | Multi-joint + safe envelope | Aug 2026 | — |
| 4 | FK in Python + hardware validation | Sep 2026 | — |
| 5 | Jacobians + differential IK | Oct 2026 | — |
| 6 | Numerical IK + test suite | Nov 2026 | — |
| 7 | **ROS 2 integration** | Dec 2026 | — |
| 8 | Trajectory generation + gravity comp | Jan 2027 | — |
| 9 | **Simulation** (MuJoCo / Gazebo) | Feb 2027 | — |
| 10 | **Perception** (camera + object detection) | Apr 2027 | — |
| 11 | **Capstone**: perception-guided pick & place | Jul 2027 | — |

**Global rules:**
- Verify before optimize. If it's not measured, it's not real.
- One variable per experiment.
- Freeze means freeze — tag commits, lock configs, archive logs.
- Every phase produces a public-ready artifact: code + plot + short write-up.
